# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 02: Gestión en GitHub Web y Métodos HTTP (GET, POST, PUT, DELETE)
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Enrutamiento manual por verbo HTTP*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Especificación semántica de verbos e idempotencia* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas de métodos no permitidos (HTTP 405)*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion02.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [1]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
⏳ Instalando PHP CLI...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Oct 19 2026 15:16:58) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Enrutamiento manual por verbo HTTP

In [5]:
%%writefile app_artesano.py
from fastapi import FastAPI, HTTPException, Request, status

app = FastAPI(title="API Artesano")

almacen = {"dato": "inicial"}

@app.api_route("/api/recurso", methods=["GET", "POST", "PUT", "DELETE", "PATCH", "HEAD", "OPTIONS"])
async def procesar_recurso(request: Request):
    metodo = request.method

    if metodo == "GET":
        return {"status": "success", "metodo": "GET", "recurso": almacen}

    if metodo in ("POST", "PUT"):
        body = await request.json()
        dato = str(body.get("dato", ""))
        if not dato.strip():
            raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio")
        almacen["dato"] = dato
        return {"status": "success", "metodo": metodo, "mensaje": f"Procesado: {dato}"}

    if metodo == "DELETE":
        almacen["dato"] = ""
        return {"status": "success", "metodo": "DELETE", "mensaje": "Recurso eliminado"}

    raise HTTPException(
        status_code=status.HTTP_405_METHOD_NOT_ALLOWED,
        detail=f"Metodo {metodo} no permitido",
        headers={"Allow": "GET, POST, PUT, DELETE"}
    )

Overwriting app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [6]:
%%writefile app_ia.py
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

app = FastAPI(title="API Cientifico")

class PayloadRequest(BaseModel):
    dato: str

recurso = {"dato": "inicial", "version": 1}

@app.get("/api/recurso")
def leer_recurso():
    return {"status": "success", "metodo": "GET", "recurso": recurso}

@app.post("/api/recurso", status_code=status.HTTP_201_CREATED)
def crear_recurso(payload: PayloadRequest):
    if not payload.dato.strip():
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio")
    recurso["dato"] = payload.dato.strip()
    recurso["version"] += 1
    return {"status": "success", "metodo": "POST", "mensaje": f"Procesado: {recurso['dato']}", "version": recurso["version"]}

@app.put("/api/recurso")
def actualizar_recurso(payload: PayloadRequest):
    if not payload.dato.strip():
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio")
    recurso["dato"] = payload.dato.strip()
    return {"status": "success", "metodo": "PUT", "mensaje": f"Actualizado: {recurso['dato']}", "version": recurso["version"]}

@app.delete("/api/recurso")
def borrar_recurso():
    recurso["dato"] = ""
    return {"status": "success", "metodo": "DELETE", "mensaje": "Recurso eliminado"}


Writing app_ia.py


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas de métodos no permitidos (HTTP 405)

In [1]:
import threading
import time
import uvicorn
import httpx
import os
from app_artesano import app

# Liberar el puerto 8000 en entornos Linux (Colab) antes de iniciar
print("🧹 Liberando el puerto 8000 por si está ocupado...")
os.system("fuser -k 8000/tcp > /dev/null 2>&1")
time.sleep(0.5)

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

# Iniciamos el servidor en un hilo secundario
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

client = httpx.Client(base_url="http://127.0.0.1:8000")
print("=== AUDITORIA CRITICA WEB: SESION 02 ===\n")

# 1. Probar POST (Creación/Modificación)
r_post = client.post("/api/recurso", json={"dato": "Prueba Test Harness"})
print(f"POST /api/recurso: HTTP {r_post.status_code} - {r_post.json()}")
assert r_post.status_code == 200

# 2. Probar GET (Lectura)
r_get = client.get("/api/recurso")
print(f"GET /api/recurso:  HTTP {r_get.status_code} - {r_get.json()}")
assert r_get.status_code == 200

# 3. Probar DELETE (Eliminación)
r_del = client.delete("/api/recurso")
print(f"DELETE /api/recurso: HTTP {r_del.status_code} - {r_del.json()}")
assert r_del.status_code == 200

# 4. Probar Método No Permitido (HTTP 405)
r_patch = client.patch("/api/recurso", json={"dato": "no permitido"})
print(f"PATCH /api/recurso: HTTP {r_patch.status_code} - {r_patch.json()}")
assert r_patch.status_code == 405

print("\nPASO ✅ - ¡Todas las pruebas del Test Harness (incluyendo HTTP 405) pasaron con éxito!")

🧹 Liberando el puerto 8000 por si está ocupado...
=== AUDITORIA CRITICA WEB: SESION 02 ===

POST /api/recurso: HTTP 200 - {'status': 'success', 'metodo': 'POST', 'mensaje': 'Procesado: Prueba Test Harness'}
GET /api/recurso:  HTTP 200 - {'status': 'success', 'metodo': 'GET', 'recurso': {'dato': 'Prueba Test Harness'}}
DELETE /api/recurso: HTTP 200 - {'status': 'success', 'metodo': 'DELETE', 'mensaje': 'Recurso eliminado'}
PATCH /api/recurso: HTTP 405 - {'detail': 'Metodo PATCH no permitido'}

PASO ✅ - ¡Todas las pruebas del Test Harness (incluyendo HTTP 405) pasaron con éxito!
